# Module 4 — Grounding Agents in Telecom Knowledge (RAG)

**NetOps Co. · CELL-031A · ~40 minutes**

In Module 1 you watched a frontier model invent a root cause for CELL-031A. It had never seen
your incident history, so it produced something plausible instead of something true.

RAG fixes that — not by making the model smarter, but by putting the right document in front of it
before it answers.

Four steps: **chunk → embed → retrieve → augment.** No vector-database internals required.

## Setup — about 60 seconds

Run this once per session. Colab gives you a fresh machine each time, so the clone and
install have to happen again — that is normal, not a mistake.

**Your API key.** Click the 🔑 key icon in the left sidebar, add a secret named
`GEMINI_API_KEY`, and toggle *Notebook access* on. Get a free key at
[aistudio.google.com](https://aistudio.google.com) — no credit card.

Never paste a key into a cell. Notebooks get shared, and the key goes with them.

In [ ]:
!git clone -q https://github.com/telcobytes/netops-genai-course.git 2>/dev/null || (cd netops-genai-course && git pull -q)
!pip install -q google-genai pydantic

import sys, os
sys.path.append('/content/netops-genai-course/data')

# Key from Colab secrets, with a local fallback so this notebook also runs
# in plain Jupyter.
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    if not os.environ.get('GEMINI_API_KEY'):
        import getpass
        os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')

print('Key loaded:', bool(os.environ.get('GEMINI_API_KEY')))
print('Module 4 — RAG')

### Sanity check — no API key needed

`mock_tools.py` is pure standard library. If this prints numbers, your environment is
working and the rest of the notebook will run.

In [ ]:
import mock_tools
summary = mock_tools.get_cell_kpis('CELL-031A')
print('window     :', summary['window_start'], '->', summary['window_end'])
print('samples    :', summary['sample_count'])
print('rolling avg:', summary['rolling_avg'])
for c in summary['thresholds_crossed']:
    print(f"  CROSSED  {c['metric']} = {c['value']} ({c['comparison']} {c['threshold']})")

---
## 1. Chunk — and why the split points matter

We split on blank lines, not on character count. A runbook step or a 3GPP table sliced in half
retrieves as nonsense.

Structure-aware chunking is boring, and it is most of the quality.

In [ ]:
sys.path.append('/content/netops-genai-course/module04-rag')
from rag_pipeline import load_and_chunk_knowledge_base, retrieve, embed_with_gemini_api
import nb_viz
from IPython.display import HTML, display

chunks = load_and_chunk_knowledge_base()
print(f'{len(chunks)} chunks from {len(set(c["source"] for c in chunks))} documents')
for c in chunks[:3]:
    print(f"\n[{c['source']}]\n{c['text'][:150]}…")

---
## 2 & 3. Embed and retrieve — retrieval is *ranking*, not lookup

This is the part people get wrong. There is no exact match happening. Every chunk is scored
against the question and sorted.

Which means the failure mode is **not** "nothing found". It is **"the wrong document won"** —
and that failure is silent unless you look at the scores.

In [ ]:
QUESTION = 'Why does CELL-031A keep congesting in the evenings, and how was it fixed before?'

vectors = embed_with_gemini_api([c['text'] for c in chunks])
qv = embed_with_gemini_api([QUESTION])[0]

def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na = sum(x*x for x in a) ** 0.5
    nb = sum(y*y for y in b) ** 0.5
    return dot / (na * nb) if na and nb else 0.0

scored = sorted(zip(chunks, vectors), key=lambda p: -cosine(qv, p[1]))[:6]
display(HTML(nb_viz.score_bars(
    [(c['source'], cosine(qv, v), c['text']) for c, v in scored],
    caption=f'Top 6 chunks for: {QUESTION}')))

> Look at the **gap** between first and second place. A wide gap means the retriever is confident.
> A narrow gap means two documents look alike to it — and that is exactly where it will
> eventually pick the wrong one.

> This bar chart is the single best reason to run this module in a notebook. In a terminal you
> see which document won. Here you see *by how much*, which is what tells you whether to trust it.

---
## 4. Augment — grounded vs ungrounded, side by side

Same question, same model. The only difference is whether two retrieved chunks were pasted in first.

In [ ]:
from llm_client import call_llm

ungrounded = call_llm([{'role':'user','content': QUESTION}])

top = retrieve(QUESTION, chunks, k=2)
context = '\n\n'.join(f"[{c['source']}]\n{c['text']}" for c in top)
grounded = call_llm([{'role':'user','content':
    f'Answer using ONLY the context below.\n\nCONTEXT:\n{context}\n\nQUESTION: {QUESTION}'}])

print('── UNGROUNDED ' + '─'*50)
print(ungrounded[:700])
print('\n── GROUNDED ' + '─'*52)
print(grounded[:700])
print('\nsources:', [c['source'] for c in top])

> The ungrounded answer is not *worse written*. It is well written and wrong.
> That is what makes it dangerous, and why a wrong grounded answer is a better problem to have:
> it is a **data** problem you can fix by editing a document, not a model problem you can't.

---
## Your turn

1. Ask something the knowledge base does **not** cover — a VoLTE codec question, say — and watch
   what comes back anyway. Retrieval always returns its top *k*; it never returns nothing.
2. Write a query that describes symptoms shared by two incidents, and see which one ranks first.
3. Set `k=5` and check whether the extra context helps or just adds noise.

In [ ]:
# Your turn
Q = 'slow speeds in the evening near SITE-031, but PRB utilization looks normal'
sc = sorted(zip(chunks, vectors), key=lambda p: -cosine(embed_with_gemini_api([Q])[0], p[1]))[:5]
display(HTML(nb_viz.score_bars([(c['source'], cosine(embed_with_gemini_api([Q])[0], v), c['text'])
                                for c, v in sc], caption=Q)))

---
**Next:** everything so far is one prompt, one answer. Module 5 is where the model starts
deciding what to do next on its own.